#Initializing AnnData

In [2]:
!pip install anndata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 96.0 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
import anndata as ad
from scipy.sparse import csr_matrix
print(ad.__version__)

0.12.10


/tmp/ipykernel_2771/2319192822.py:5: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print(ad.__version__)


In [4]:
counts = csr_matrix(np.random.poisson(1, size=(100, 2000)), dtype=np.float32)
adata = ad.AnnData(counts)
adata

AnnData object with n_obs × n_vars = 100 × 2000

In [5]:
adata.X

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 126566 stored elements and shape (100, 2000)>

In [6]:
adata.obs_names = [f"Cell_{i:d}" for i in range(adata.n_obs)]
adata.var_names = [f"Gene_{i:d}" for i in range(adata.n_vars)]
print(adata.obs_names[:10])

Index(['Cell_0', 'Cell_1', 'Cell_2', 'Cell_3', 'Cell_4', 'Cell_5', 'Cell_6',
       'Cell_7', 'Cell_8', 'Cell_9'],
      dtype='object')


In [7]:
adata[["Cell_1", "Cell_10"], ["Gene_5", "Gene_1900"]]

View of AnnData object with n_obs × n_vars = 2 × 2

# Adding aligned metadata

In [8]:
ct = np.random.choice(["B", "T", "Monocyte"], size=(adata.n_obs,))
adata.obs["cell_type"] = pd.Categorical(ct)  # Categoricals are preferred for efficiency
adata.obs

,cell_type
Cell_0,B
Cell_1,B
Cell_2,B
Cell_3,Monocyte
Cell_4,B
...,...
Cell_95,T
Cell_96,B
Cell_97,Monocyte
Cell_98,Monocyte


In [9]:
adata

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type'

In [10]:
bdata = adata[adata.obs.cell_type == "B"]
bdata

View of AnnData object with n_obs × n_vars = 28 × 2000
    obs: 'cell_type'

# Observation/variable-level matrices

In [11]:
adata.obsm["X_umap"] = np.random.normal(0, 1, size=(adata.n_obs, 2))
adata.varm["gene_stuff"] = np.random.normal(0, 1, size=(adata.n_vars, 5))
adata.obsm

AxisArrays with keys: X_umap

In [12]:
adata

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type'
    obsm: 'X_umap'
    varm: 'gene_stuff'

# Unstructured metadata

In [13]:
adata.uns["random"] = [1, 2, 3]
adata.uns

OrderedDict([('random', [1, 2, 3])])

# Layers

In [14]:
adata.layers["log_transformed"] = np.log1p(adata.X)
adata

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type'
    uns: 'random'
    obsm: 'X_umap'
    varm: 'gene_stuff'
    layers: 'log_transformed'

# Conversion to DataFrames

In [15]:
adata.to_df(layer="log_transformed")

,Gene_0,Gene_1,Gene_2,Gene_3,Gene_4,Gene_5,Gene_6,Gene_7,Gene_8,Gene_9,...,Gene_1990,Gene_1991,Gene_1992,Gene_1993,Gene_1994,Gene_1995,Gene_1996,Gene_1997,Gene_1998,Gene_1999
Cell_0,0.000000,0.693147,0.000000,0.000000,0.693147,0.000000,1.098612,1.098612,0.000000,1.098612,...,1.386294,0.000000,1.386294,0.000000,0.000000,0.000000,0.693147,0.693147,0.000000,0.000000
Cell_1,0.693147,1.386294,0.693147,1.098612,1.098612,0.693147,0.693147,0.000000,0.000000,0.693147,...,1.386294,1.098612,1.098612,0.693147,0.693147,1.098612,0.693147,0.693147,0.693147,0.000000
Cell_2,1.609438,1.386294,1.098612,0.000000,0.000000,0.693147,0.000000,0.693147,0.000000,1.098612,...,0.000000,0.000000,0.693147,0.000000,0.693147,1.098612,0.693147,1.098612,1.609438,0.693147
Cell_3,0.000000,0.693147,1.098612,0.693147,1.098612,0.693147,1.098612,0.000000,0.693147,0.693147,...,0.693147,0.000000,0.693147,0.693147,0.000000,1.098612,0.693147,0.000000,0.693147,0.000000
Cell_4,0.693147,1.098612,0.000000,0.000000,0.000000,1.386294,1.098612,1.098612,0.000000,0.693147,...,1.386294,0.693147,1.791759,0.000000,0.693147,0.000000,0.693147,0.000000,0.693147,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Cell_95,0.693147,0.000000,0.693147,1.098612,0.693147,1.098612,0.693147,0.693147,1.098612,0.000000,...,1.098612,0.693147,1.098612,0.693147,0.000000,1.386294,1.609438,0.693147,0.693147,1.098612
Cell_96,0.000000,0.000000,1.098612,0.693147,0.693147,0.000000,0.693147,0.000000,0.693147,0.693147,...,0.693147,1.098612,0.000000,1.386294,0.000000,0.000000,0.693147,1.098612,1.609438,1.098612
Cell_97,0.000000,1.098612,0.693147,0.000000,0.000000,0.000000,0.693147,1.098612,1.098612,0.693147,...,0.000000,1.098612,0.693147,0.000000,1.098612,1.098612,0.000000,1.098612,0.693147,0.693147
Cell_98,0.000000,0.000000,0.693147,0.000000,1.098612,1.098612,0.000000,1.098612,1.386294,0.693147,...,0.000000,1.386294,0.000000,0.000000,0.693147,0.000000,0.693147,1.098612,0.000000,1.098612


# Writing the results to disk

In [16]:
adata.write('my_results.h5ad', compression="gzip")

In [17]:
!h5ls 'my_results.h5ad'

/bin/bash: line 1: h5ls: command not found


# Views and copies

In [18]:
obs_meta = pd.DataFrame({
        'time_yr': np.random.choice([0, 2, 4, 8], adata.n_obs),
        'subject_id': np.random.choice(['subject 1', 'subject 2', 'subject 4', 'subject 8'], adata.n_obs),
        'instrument_type': np.random.choice(['type a', 'type b'], adata.n_obs),
        'site': np.random.choice(['site x', 'site y'], adata.n_obs),
    },
    index=adata.obs.index,    # these are the same IDs of observations as above!
)

In [19]:
adata = ad.AnnData(adata.X, obs=obs_meta, var=adata.var)

In [20]:
print(adata)

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'time_yr', 'subject_id', 'instrument_type', 'site'


In [21]:
adata

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'time_yr', 'subject_id', 'instrument_type', 'site'

In [22]:
adata[:5, ['Gene_1', 'Gene_3']]

View of AnnData object with n_obs × n_vars = 5 × 2
    obs: 'time_yr', 'subject_id', 'instrument_type', 'site'

In [23]:
adata_subset = adata[:5, ['Gene_1', 'Gene_3']].copy()

In [24]:
print(adata[:3, 'Gene_1'].X.toarray().tolist())
adata[:3, 'Gene_1'].X = [0, 0, 0]
print(adata[:3, 'Gene_1'].X.toarray().tolist())

[[1.0], [3.0], [3.0]]
[[0.0], [0.0], [0.0]]


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:630: FutureWarning: Setting element `.X` of view of `AnnData` object will obey copy-on-write semantics in the next minor release. 
  if self._handle_view_X_cow(value):
/tmp/ipykernel_2771/2822107891.py:2: ImplicitModificationWarning: Modifying `X` on a view results in data being overridden
  adata[:3, 'Gene_1'].X = [0, 0, 0]


In [25]:
adata_subset = adata[:3, ['Gene_1', 'Gene_2']]

In [26]:
adata_subset

View of AnnData object with n_obs × n_vars = 3 × 2
    obs: 'time_yr', 'subject_id', 'instrument_type', 'site'

In [27]:
adata_subset.obs['foo'] = range(3)

/tmp/ipykernel_2771/2955902014.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_subset.obs['foo'] = range(3)


In [28]:
adata_subset

AnnData object with n_obs × n_vars = 3 × 2
    obs: 'time_yr', 'subject_id', 'instrument_type', 'site', 'foo'

In [29]:
adata[adata.obs.time_yr.isin([2, 4])].obs.head()

,time_yr,subject_id,instrument_type,site
Cell_0,2,subject 4,type b,site y
Cell_1,2,subject 1,type a,site y
Cell_2,2,subject 2,type b,site x
Cell_3,2,subject 8,type a,site x
Cell_4,4,subject 1,type b,site x


# Partial reading of large data

In [31]:
adata = ad.read_h5ad('my_results.h5ad', backed='r')

In [32]:
adata.isbacked

True

In [33]:
adata.filename

PosixPath('my_results.h5ad')

In [34]:
adata.file.close()